# R-tag pipeline — single run walkthrough

KB331 sample filter. Run top-to-bottom from `python_code/`. Docs: [README](../README.md) · [CLAUDE](../CLAUDE.md).

| Step | Call |
|------|------|
| 2.x | `identify` → resonator list |
| 3 | `prep_resonator_ppd` + orientation shift |
| 4 | `prep_rteg_in_frame` |
| 5.1 | `collect_geometry_roles` |
| 5.2 | `classify_nodes` |
| 5.3 | `build_mte_extensions` |
| 5.4 | `build_mte_pad_routes` (center pad only) |
| 6.1 | `build_mbe_extensions` (`collar_extend` only) |
| 6.2 | `build_mbe_bodies` (`collar_extend` only) |


In [ ]:
from __future__ import annotations
import io
import sys
from contextlib import redirect_stdout
from pathlib import Path
import gdstk
import pandas as pd


def resolve_python_code_root() -> Path:
    """Find python_code/ by looking for input_files/ + src/."""
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "input_files").is_dir() and (candidate / "src").is_dir():
            return candidate
    return here


ROOT = resolve_python_code_root()
SRC = ROOT / "src"
INPUT_FILES = ROOT / "input_files"
DRAFT = ROOT / "draft_output"
ORIGNAL_RTEG = DRAFT / "original_rteg"

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

DRAFT.mkdir(parents=True, exist_ok=True)

print(f"Working directory: {ROOT}")
print(f"Input files:       {INPUT_FILES}")
print(f"Draft output:      {DRAFT}")
print(f"Source code:       {SRC}")

## Define Inputs Here For Running This Notebook

In [ ]:
FILTER = INPUT_FILES / "KB331_N_01.gds" # input filter GDS file
FRAME = INPUT_FILES / "KB331_N_Frame.gds" # die frame
PPD = INPUT_FILES / "GSG_frame.gds" # GSG ppd frame
LAYERMAP = INPUT_FILES / "layermap" # Skyworks layer map

## 1. Process Inputs

In [ ]:
# Ensure all referenced input files exist; abort on missing inputs

input_files = {
    "Filter layout": FILTER,
    "Frame template": FRAME,
    "Probe device": PPD,
    "Layer table": LAYERMAP,
}

input_roles = {
    "Filter layout": "Clean hierarchical filter GDS — resonators + connect metal",
    "Frame template": None,
    "Probe device": "ppd_1port — GSG pad reference",
    "Layer table": "Skyworks name → GDS (layer, datatype)",
}

rows = []
frame_size_str = "unknown size"

for label, path in input_files.items():
    if not path.exists():
        raise FileNotFoundError(f"Missing required input: {label} ({path})")
    size = f"{path.stat().st_size:,} B"
    rows.append({"file": label, "path": path.name, "exists": True, "size": size, "role": input_roles[label]})

frame_lib = gdstk.read_gds(FRAME)
frame_cell = frame_lib.top_level()[0]
frame_bb = frame_cell.bounding_box()
if frame_bb is not None:
    (fx0, fy0), (fx1, fy1) = frame_bb
    frame_size_str = f"{fx1 - fx0:.1f}×{fy1 - fy0:.1f} µm"

for row in rows:
    if row["file"] == "Frame template":
        row["role"] = f"{frame_size_str} GSG probe frame (six BAW_MB2 pads)"

display(pd.DataFrame(rows))


## 2. Selection

2.1 start with layermap definitions

2.2 inspect layer references

2.3 start identifying resonators from filter GDS file

### 2.1 `layermap.py` — layer name lookups

Loads the Skyworks layermap file from `LAYERMAP` (defined above). Maps names like `BAW_MBE` to GDS `(layer, datatype)` pairs.

In [ ]:
from src.layermap import load_layermap

layermap = load_layermap(LAYERMAP)
print(f"Loaded {len(layermap)} layers from {LAYERMAP.name}")

for name in ("BAW_MBE", "BAW_MTE", "BAW_MB2", "BAW_EDGE"):
    layer, dt = layermap.pair(name)
    print(f"  {name:12s} -> GDS ({layer}, {dt})")

### 2.2 `inspect_refs.py` — hierarchy sanity check

Lists placed references in the filter GDS: resonators, vias, connect cells. Useful when instance names did not survive export.

In [ ]:
from src.inspect_refs import inspect_gds

buf = io.StringIO()
with redirect_stdout(buf):
    inspect_gds(FILTER)

# 
print(buf.getvalue()[:2000])
if len(buf.getvalue()) > 2000:
    print("... (truncated)")

### 2.3 `separate.py` — resonator and vtb identification

Finds placed resonators (masters: `series`, `shunt`, `rcap`, `mimcap`) and `vtb` vias under the filter parent cell. Returns dataframe-ready rows via `identify(FILTER)`.

In [ ]:
from src.separate import identify

identification = identify(FILTER)

parent = identification.parent
filter_lib = identification.library
res_list = identification.resonators
vias = identification.vias

res_df = pd.DataFrame(identification.resonator_rows()) #DF of resonators 
via_df = pd.DataFrame(identification.via_rows()) #DF of VIAs

print(f"Parent cell: {parent}")
print(f"Resonators: {len(res_list)}  |  Vias: {len(vias)}")

res_df

In [ ]:
via_df

## 3. Separation
For each resonator found place it together with the GSG ppd frame in the center

### 3 — PPD + orientation placement

**Calls:** `prep_resonator_ppd` → `preserved_collars_at_shift` → `analyze_orientation`

Centers each resonator on the GSG template and applies clearance + orientation shift.


In [ ]:
from IPython.display import HTML, display

from src.prep_resonator_ppd import (
    MIN_RELEASE_HOLE_CLEARANCE_UM,
    assemblies_summary_df,
    prep_resonator_ppd,
)

ppd_assemblies = prep_resonator_ppd(
    res_df,
    res_list,
    PPD,
    identification=identification,
    layermap=layermap,
)
display(assemblies_summary_df(ppd_assemblies))


simply preview of the resonator + ppd GSG placement within the notebook directly 

In [ ]:
# # Display/Preview of frame within this notebook

# _preview_cols = 4
# _preview_items = "\n".join(
#     f'<div class="ppd-preview-item">'
#     f'<div class="ppd-preview-label">[{asm.index}] {asm.inst_name} ({asm.res_type})</div>'
#     f'{preview_assembly_svg(asm)}'
#     f"</div>"
#     for asm in ppd_assemblies
# )
# display(
#     HTML(
#         f"""
# <style>
# .ppd-preview-grid {{
#   display: grid;
#   grid-template-columns: repeat({_preview_cols}, minmax(0, 1fr));
#   gap: 8px;
#   margin-top: 8px;
# }}
# .ppd-preview-item {{
#   border: 1px solid #ccc;
#   padding: 4px;
#   text-align: center;
#   overflow: hidden;
# }}
# .ppd-preview-label {{
#   font-size: 11px;
#   margin-bottom: 4px;
# }}
# .ppd-preview-item svg {{
#   width: 100%;
#   height: auto;
#   display: block;
# }}
# </style>
# <div class="ppd-preview-grid">
# {_preview_items}
# </div>
# """
#     )
# )

## 4. Setting Up

place the ppd+resonator combo now into the original die frame.

for the width (x axis) leave 4% margin in both sides

for the height (y axis) leave 7% margin in both sides

### 4 — Die frame placement

**Call:** `prep_rteg_in_frame` → `rteg_assemblies`

Places each PPD+resonator assembly into the die frame template.


In [ ]:
from src.prep_rteg_frame import (
    assemblies_summary_df,
    prep_rteg_in_frame,
    preview_assembly_svg,
)

rteg_assemblies = prep_rteg_in_frame(ppd_assemblies, FRAME)
rteg_files_df = assemblies_summary_df(rteg_assemblies)

print(f"Built {len(rteg_assemblies)} RTEG frame assemblies")
display(rteg_files_df)

In [ ]:
# from IPython.display import HTML, display

# _preview_cols = 4
# _preview_items = "\n".join(
#     f'<div class="rteg-preview-item">'
#     f'<div class="rteg-preview-label">[{asm.index}] {asm.inst_name} ({asm.ppd_assembly.res_type})</div>'
#     f'{preview_assembly_svg(asm)}'
#     f"</div>"
#     for asm in rteg_assemblies
# )
# display(
#     HTML(
#         f"""
# <style>
# .rteg-preview-grid {{
#   display: grid;
#   grid-template-columns: repeat({_preview_cols}, minmax(0, 1fr));
#   gap: 8px;
#   margin-top: 8px;
# }}
# .rteg-preview-item {{
#   border: 1px solid #ccc;
#   padding: 4px;
#   text-align: center;
#   overflow: hidden;
# }}
# .rteg-preview-label {{
#   font-size: 11px;
#   margin-bottom: 4px;
# }}
# .rteg-preview-item svg {{
#   width: 100%;
#   height: auto;
#   display: block;
# }}
# </style>
# <div class="rteg-preview-grid">
# {_preview_items}
# </div>
# """
#     )
# )

In [ ]:
from src.export_gds import export_gds, export_summary_df

rteg_export_df = export_summary_df(
    export_gds(
        rteg_assemblies,
        ORIGNAL_RTEG,
        layermap=layermap,
        parent=parent,
        stage="framed",
    )
)

print(f"Exported {len(rteg_export_df)} GDS files to {ORIGNAL_RTEG}")
print("Layer names: open each .gds with its matching .lyp in KLayout")
display(rteg_export_df)


========================================================

## 5. Routing

Build a DRC-clean RTEG: collect geometry → classify GSG → draw MTE extensions → route to pad (if center-facing) → (later) union ground and carve pockets.

| Step | Module | Output |
|------|--------|--------|
| 5.1 | `rteg_collect` | `all_roles` |
| 5.2 | `rteg_classify` | `all_classify` |
| 5.3 | `rteg_mte_extensions` | `all_mte` |
| 5.4 | `rteg_mte_route` | stretched `routed_net` for indices 4 & 6 |
| 6.1 | `rteg_mbe_extensions` | `all_mbe` for indices 0, 2, 5 & 7 |


### 5.1 — Collect geometry roles

**Call:** `collect_geometry_roles` → `all_roles[index]`

Pulls ground plates, preserved filter MBE/MTE, release holes, frame boundary, and body MTE per resonator.

**Preserved MTE:** `connectMTE` polygons in the resonator window; series parts also get the body-side collar tab that touches the stadium (e.g. index 6 → areas 911 + 5191).

**Config:** `RtegCollectConfig` — key fields: `preserved_overlap_margin_um` (10), `stadium_collar_area_um2` (2500), `max_body_interface_collar_area_um2` (2000). See `rteg_collect.py` for full list.


In [ ]:
from src.rteg_collect import (
    RtegCollectConfig,
    collect_geometry_roles,
    geometry_roles_summary_table,
)

COLLECT_CONFIG = RtegCollectConfig()

all_roles: dict[int, object] = {}
collect_rows: list[dict[str, object]] = []
collect_detail_rows: list[dict[str, object]] = []

for idx in range(len(identification.resonators)):
    res = identification.resonators[idx]
    roles = collect_geometry_roles(
        rteg_assemblies[idx],
        res,
        identification,
        layermap,
        config=COLLECT_CONFIG,
    )
    all_roles[idx] = roles
    counts = roles.group_counts()
    collect_rows.append(
        {
            "index": idx,
            "inst_name": roles.inst_name,
            "res_type": res.res_type,
            **{k: counts[k] for k in sorted(counts)},
            "total_polygons": sum(counts.values()),
        }
    )
    collect_detail_rows.extend(geometry_roles_summary_table(roles))

collect_overview_df = pd.DataFrame(collect_rows).sort_values("index")
collect_detail_df = pd.DataFrame(collect_detail_rows)

print(f"Collected geometry for {len(all_roles)} resonators\n")
display(collect_overview_df)
print("\nPolygon detail (all resonators):")
display(collect_detail_df)

### 5.2 — Classify GSG nodes

**Call:** `classify_nodes` → `all_classify[index]`

**`mte_route_target`:** `center_pad` when the **mouth tab** (smallest connectMTE overlapping body, not stadium) is closer to the center pad than the **body bbox center** is: `dist(collar, pad) < dist(body, pad)`. Else `collar_extend`.


In [ ]:
from src.rteg_classify import classify_nodes, classification_summary_table
from src.rteg_collect import collect_orientation_inputs

all_classify: dict[int, object] = {}
classify_overview_rows: list[dict[str, object]] = []
classify_detail_rows: list[dict[str, object]] = []

for idx, roles in all_roles.items():
    res = identification.resonators[idx]
    orientation = collect_orientation_inputs(
        rteg_assemblies[idx],
        res,
        identification,
        layermap,
        ground_plates=roles.ground_plates,
        config=COLLECT_CONFIG,
    )
    classification = classify_nodes(
        roles.ground_plates,
        roles.preserved,
        orientation=orientation,
        res_type=res.res_type,
    )
    all_classify[idx] = classification
    collar = classification.collar_orientation
    by_band = classification.by_band()
    classify_overview_rows.append(
        {
            "index": idx,
            "inst_name": roles.inst_name,
            "res_type": res.res_type,
            "method": classification.method,
            "mte_route_target": classification.mte_route_target,
            "mte_faces_center": collar.mte_faces_center,
            "signal_terminal": classification.signal_terminal,
            "signal_drawable": classification.signal_drawable,
            "collar_axis": collar.axis,
            "facing_pad": collar.facing_pad,
            "top": by_band["top"].net,
            "center": by_band["center"].net,
            "bottom": by_band["bottom"].net,
            "note": classification.note,
        }
    )
    classify_detail_rows.extend(
        classification_summary_table(
            classification,
            index=idx,
            inst_name=roles.inst_name,
            res_type=res.res_type,
        )
    )

classify_overview_df = pd.DataFrame(classify_overview_rows).sort_values("index")
classify_df = pd.DataFrame(classify_detail_rows)

print(f"Classified {len(all_classify)} resonators\n")
display(classify_overview_df)

for idx, classification in all_classify.items():
    assert classification.method == "orientation"
    assert classification.mte_route_target in ("center_pad", "collar_extend")
    assert classification.signal_drawable == bool(all_roles[idx].preserved.mte)
    if classification.mte_route_target == "center_pad":
        assert classification.by_band()["center"].net == "signal"

print(f"\nOrientation classification checks passed for all {len(all_classify)} resonators")


### 5.3 — MTE extensions

**Call:** `build_mte_extensions(all_roles, layermap, mte_cfg)` → one extension per resonator.

Pipeline per index (no 5.2 input):

1. **`select_extension_collar`** — small mouth tab touching stadium (not die-wide bus).
2. **`find_outward_lip_ab`** — best outward lip edge → corners A, B.
3. **`draw_lip_extension`** — rectangle: merge into collar, extrude 14 µm outward.
4. **DONE** — `is_connected == True` when overlap + mouth checks pass.

**Golden baseline:** index 6 — collar 911, stadium 5191, extension on 911.

**Config:** `MteBuildConfig` in code cell — `collar_extension_um` (14), `collar_merge_inset_um` (4). Full list in `rteg_mte_extensions.py`.

**Tables:** `n_preserved_mte`, `is_connected`, `collar_overlap_um2`; intercept breakdown shows A/B and mouth span.


In [ ]:
# DRAW MTE extensions for all resonators. 
# Disregard the orientation or position of signal in this step

from src.rteg_mte_extensions import (
    MteBuildConfig,
    build_mte_extensions,
    mte_extensions_overview_rows,
    mte_intercept_breakdown_rows,
)

# --- 5.3 tunables (edit here) ---
mte_cfg = MteBuildConfig(
    mte_layer="BAW_MTE",
    collar_extension_um=14.0,
    collar_merge_inset_um=4.0,
    collar_touch_overlap_um=0.5,
    min_collar_overlap_um2=0.01,
    stadium_collar_area_um2=2500.0,
    stadium_edge_area_ratio=0.6,
    lip_long_edge_peak_fraction=0.15,
    lip_long_edge_min_um=8.0,
    max_overlap_fraction=0.99,
    min_merge_inset_check_um=0.5,
    boolean_precision=1e-3,
    inside_probe_half_um=0.25,
    feasible_merge_search_iterations=24,
)

all_mte = build_mte_extensions(all_roles, layermap, mte_cfg)

inst_names = {idx: roles.inst_name for idx, roles in all_roles.items()}

mte_overview_df = pd.DataFrame(
    mte_extensions_overview_rows(all_mte, inst_names=inst_names)
).sort_values("index")

display(mte_overview_df)
print(f"Drew MTE extensions for {len(all_mte)} resonators")

mte_intercept_df = pd.DataFrame(
    mte_intercept_breakdown_rows(all_mte, inst_names=inst_names)
).sort_values("index")

print("Intercept breakdown (two end-cap edges on preserved collar):")
display(mte_intercept_df)


**Call:** `export_mte_extensions_gds(rteg_assemblies, all_mte, MTE_OUT, layermap=..., mbe_extensions=all_mbe)`

One GDS per resonator under `MTE_generated_deterministic`: frame + MTE (5/0) + MBE (2/0) when step 6 has run. Re-run export after 5.4 and again after 6.1.


In [ ]:
# 5.3 export — writes frame + MTE (+ MBE when all_mbe exists) to GDS.
# Re-run after 5.4 or 6.1 to refresh the same files.

from src.export_gds import export_summary_df
from src.rteg_mte_extensions import export_mte_extensions_gds

MTE_OUT = DRAFT / "MTE_generated_deterministic"
mte_export_df = export_summary_df(
    export_mte_extensions_gds(
        rteg_assemblies,
        all_mte,
        MTE_OUT,
        layermap=layermap,
        parent=parent,
    )
)

print(f"Exported {len(mte_export_df)} GDS files to {MTE_OUT}")
display(mte_export_df)

### 5.4 — Stretch MTE to center signal pad

**Call:** `build_mte_pad_routes(all_roles, all_classify, all_mte, layermap, mte_route_cfg)`

Only `center_pad` resonators (KB331 indices **4**, **6**). Stretches the 5.3 outer cap to the signal pad facing edge (full pad height, `pad_touch_overlap_um` into pad). Collar mouth vertices stay fixed. Re-exports GDS below.

**Config:** `MteRouteConfig` — `pad_touch_overlap_um` (0.5), `min_pad_overlap_um2` (0.01).


In [ ]:
from src.export_gds import export_summary_df
from src.rteg_mte_extensions import export_mte_extensions_gds
from src.rteg_mte_route import (
    MteRouteConfig,
    build_mte_pad_routes,
    mte_route_overview_rows,
)

# --- 5.4 tunables (edit here) ---
mte_route_cfg = MteRouteConfig(
    mte_layer="BAW_MTE",
    pad_touch_overlap_um=0.5,
    min_pad_overlap_um2=0.01,
    boolean_precision=1e-3,
    inside_probe_half_um=0.25,
)

all_mte = build_mte_pad_routes(
    all_roles, all_classify, all_mte, layermap, mte_route_cfg
)

mte_route_df = pd.DataFrame(
    mte_route_overview_rows(all_mte, all_classify, inst_names=inst_names)
).sort_values("index")

display(mte_route_df)
routed_count = int(mte_route_df["routed_to_pad"].sum())
print(f"Routed {routed_count} resonator(s) to center signal pad")


MTE_OUT = DRAFT / "MTE_generated_deterministic"
mte_export_df = export_summary_df(
    export_mte_extensions_gds(
        rteg_assemblies,
        all_mte,
        MTE_OUT,
        layermap=layermap,
        parent=parent,
    )
)

print(f"Exported {len(mte_export_df)} GDS files to {MTE_OUT}")
display(mte_export_df)


## Step 6 — MBE routing (`mte_faces_center = false`)

When MTE does not face center, route **signal on MBE** from the center pad to the preserved MBE collar.

### 6.1 — MBE pad-to-collar connection

Straight-line connection from the center pad top-right / bottom-right corners to the MBE collar (aiming at the collar vertex nearest release holes). Only for resonators where **`mte_faces_center` is false** (indices 0, 2, 5, 7 on KB331). Exports frame + MTE (5/0) + MBE (2/0) to `MTE_generated_deterministic`.


In [ ]:
# MBE pad-to-collar connection for resonators where mte_faces_center = false.

from src.export_gds import export_summary_df
from src.rteg_mbe_extensions import (
    MbeConnectionConfig,
    build_mbe_extensions,
    mbe_extensions_overview_rows,
)
from src.rteg_mte_extensions import export_mte_extensions_gds

mbe_cfg = MbeConnectionConfig()

all_mbe = build_mbe_extensions(all_roles, all_classify, layermap, mbe_cfg)

mbe_overview_df = pd.DataFrame(
    mbe_extensions_overview_rows(all_mbe, inst_names=inst_names)
).sort_values("index")

display(mbe_overview_df)
mbe_drawn = int(mbe_overview_df["n_extensions"].sum())
print(f"Drew MBE connections for {mbe_drawn} resonators (mte_faces_center = false)")

MTE_OUT = DRAFT / "MTE_generated_deterministic"
mte_export_df = export_summary_df(
    export_mte_extensions_gds(
        rteg_assemblies,
        all_mte,
        MTE_OUT,
        layermap=layermap,
        parent=parent,
        mbe_extensions=all_mbe,
    )
)

print(f"Exported {len(mte_export_df)} MTE + MBE GDS files to {MTE_OUT}")
display(mte_export_df)


## Step 6.2 — MBE ground body (`collar_extend` only)

When preserved MTE did **not** face the center signal pad (indices 0, 2, 5, 7 on KB331):

1. Copy the 5.3 MTE extension, keep the outer half, shift 3.5 µm toward the filler (overlaps MTE + carved filler).
2. Carve the step-4 width filler around the resonator stadium with DRC clearance.
3. Export the cap and carved filler as **separate** `BAW_MBE` polygons (not boolean-merged).

Re-export GDS after 6.2 to include the carved filler (replaces the raw step-4 rectangle).

In [ ]:
# MBE ground body for resonators where mte_route_target = collar_extend.

from src.export_gds import export_summary_df
from src.rteg_mbe_body import (
    MbeBodyConfig,
    build_mbe_bodies,
    mbe_body_overview_rows,
)
from src.rteg_mte_extensions import export_mte_extensions_gds

mbe_body_cfg = MbeBodyConfig()

all_mbe_body = build_mbe_bodies(
    all_roles,
    all_classify,
    all_mte,
    all_mbe,
    layermap,
    mbe_body_cfg,
)

mbe_body_overview_df = pd.DataFrame(
    mbe_body_overview_rows(all_mbe_body, inst_names=inst_names)
).sort_values("index")

display(mbe_body_overview_df)
body_drawn = int(mbe_body_overview_df["n_pieces"].sum())
print(f"Drew MBE ground body for {body_drawn} polygon piece(s) (collar_extend)")

MTE_OUT = DRAFT / "MTE_generated_deterministic"
mte_export_df = export_summary_df(
    export_mte_extensions_gds(
        rteg_assemblies,
        all_mte,
        MTE_OUT,
        layermap=layermap,
        parent=parent,
        mbe_extensions=all_mbe,
        mbe_bodies=all_mbe_body,
    )
)

print(f"Exported {len(mte_export_df)} MTE + MBE GDS files to {MTE_OUT}")
display(mte_export_df)